# 06 — Hyperparameter Tuning

**Goal:** Fine-tune our models to squeeze out the maximum possible performance without overfitting. We will use `TimeSeriesSplit` to ensure we don't accidentally leak future data into the past during cross-validation.

## 1. Imports & Data Loading

In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
import joblib
import warnings
warnings.filterwarnings('ignore')

FEATURED_PATH = '../data/processed/featured.csv'
MODELS_DIR = '../models/'

df = pd.read_csv(FEATURED_PATH, parse_dates=['datetime'], index_col='datetime')
TARGET = 'Global_active_power'
FEATURES = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_weekend', 
            'lag_1h', 'lag_24h', 'lag_48h', 'rolling_mean_6h', 'rolling_mean_24h', 'unmetered_energy']

split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

Train shape: (13514, 11), Test shape: (3379, 11)


## 2. TimeSeriesSplit (Cross-Validation for the Future)

**What:** Instead of standard K-Fold cross-validation (which shuffles data randomly), we use `TimeSeriesSplit`.
**Why:** If we shuffle time-series data, a model might train on December 2008 and test on January 2008. This is time travel. `TimeSeriesSplit` creates "folds" where the train set always strictly precedes the validation set.

In [2]:
tscv = TimeSeriesSplit(n_splits=3)

## 3. Tuning Ridge Regression

**What:** Testing different `alpha` penalties.
**Why:** To find the perfect balance between letting the model learn from features and punishing it for relying too heavily on any single feature.

In [3]:
print('Tuning Ridge Regression...')
ridge_param_grid = {'alpha': [0.1, 1.0, 10.0, 100.0]}
ridge_search = RandomizedSearchCV(Ridge(), ridge_param_grid, n_iter=4, cv=tscv, 
                                  scoring='neg_mean_absolute_error', random_state=42)
ridge_search.fit(X_train, y_train)
best_ridge = ridge_search.best_estimator_
print(f'Best Ridge Params: {ridge_search.best_params_}')

Tuning Ridge Regression...
Best Ridge Params: {'alpha': 100.0}


## 4. Tuning Random Forest

**What:** Adjusting the number of trees (`n_estimators`) and how deep they are allowed to grow (`max_depth`).
**Why:** A tree that is too deep memorizes the data (overfitting). A tree that is too shallow learns nothing (underfitting). We need the sweet spot.

In [4]:
print('Tuning Random Forest...')
rf_param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, 15]
}
rf_search = RandomizedSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), 
                               rf_param_grid, n_iter=4, cv=tscv, 
                               scoring='neg_mean_absolute_error', random_state=42)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_
print(f'Best RF Params: {rf_search.best_params_}')

Tuning Random Forest...
Best RF Params: {'n_estimators': 50, 'max_depth': 10}


## 5. Tuning XGBoost

**What:** Adjusting trees, depth, and `learning_rate`.
**Why:** In XGBoost, learning rate determines how aggressively each tree tries to fix the mistakes of the previous tree. A high learning rate is fast but reckless; a low learning rate is safe but requires hundreds of trees to finish the job.

In [5]:
print('Tuning XGBoost...')
xgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7]
}
xgb_search = RandomizedSearchCV(XGBRegressor(random_state=42), xgb_param_grid, 
                                n_iter=5, cv=tscv, scoring='neg_mean_absolute_error', random_state=42)
xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_
print(f'Best XGBoost Params: {xgb_search.best_params_}')

Tuning XGBoost...
Best XGBoost Params: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05}


## 6. Final Evaluation & Saving the Best Model

**What:** Pitting all the finely-tuned models against each other on the unseen Test Set one last time.
**Why:** Cross-validation helps us tune, but the ultimate test is always a completely untouched, virgin dataset. The winner gets saved as `best_model.pkl` and goes to production!

In [6]:
def evaluate(model, name):
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f'{name} -> MAE: {mae:.4f} kW | R²: {r2:.4f}')
    return mae

ridge_mae = evaluate(best_ridge, 'Tuned Ridge')
rf_mae = evaluate(best_rf, 'Tuned Random Forest')
xgb_mae = evaluate(best_xgb, 'Tuned XGBoost')

# Dynamically save the best
models_dict = {'Ridge': (best_ridge, ridge_mae), 'Random Forest': (best_rf, rf_mae), 'XGBoost': (best_xgb, xgb_mae)}
winner_name = min(models_dict, key=lambda k: models_dict[k][1])
winner_model = models_dict[winner_name][0]

joblib.dump(winner_model, f'{MODELS_DIR}best_model.pkl')
print(f'\nWinner: {winner_name}! Saved as best_model.pkl')

Tuned Ridge -> MAE: 0.2894 kW | R²: 0.7504
Tuned Random Forest -> MAE: 0.2383 kW | R²: 0.7790
Tuned XGBoost -> MAE: 0.2330 kW | R²: 0.7910

Winner: XGBoost! Saved as best_model.pkl
